# Dyck: 4 Models + Probes

This notebook is the main task-specific entry point for the Dyck experiments in `experiment_pipeline_plan.md`.

It trains four model families on one Dyck setting, extracts hidden states, runs sufficient-statistic probes, and writes the run summaries to `results/notebooks/dyck/<task_name>/`.

Expected model set:
- `rnn`
- `lstm`
- `transformer`
- `mamba` using the official `mamba-ssm` package


## 0. Path Setup

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT


PosixPath('/home/hp_twist_shan/Research/Hidden State Evaluation')

## 1. Imports

In [2]:
import pandas as pd
import torch

from hse.experiments.dyck import (
    DEFAULT_DYCK_TASKS,
    official_mamba_status,
    resolve_dyck_model_specs,
    run_dyck_suite,
)


/home/hp_twist_shan/miniforge3/envs/hse/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Experiment Settings

Default values below are a smoke configuration so `Run All` finishes quickly.

For the real experiment in the plan, switch to something closer to:

```python
TRAINING_STEPS = 10_000
BATCH_SIZE = 128
EXTRACT_EXAMPLES = 50_000
SEEDS = [0, 1, 2]
```


In [3]:
TASK_NAME = "dyck_no_noise"  # or "dyck_50_noise"
SEEDS = [0]
TRAINING_STEPS = 200
BATCH_SIZE = 32
EXTRACT_EXAMPLES = 512
EXTRACT_BATCH_SIZE = 256
LEARNING_RATE = 3e-4
EVAL_EVERY = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Keep this True if you want the notebook to enforce the official 4-model comparison.
REQUIRE_OFFICIAL_MAMBA = True
FALLBACK_TO_MAMBA_LIKE = False

RESULTS_ROOT = ROOT / "results" / "notebooks" / "dyck" / TASK_NAME
RESULTS_ROOT


PosixPath('/home/hp_twist_shan/Research/Hidden State Evaluation/results/notebooks/dyck/dyck_no_noise')

## 3. Preflight Checks

This cell fails early if official `mamba-ssm` is missing while `REQUIRE_OFFICIAL_MAMBA = True`.


In [4]:
assert TASK_NAME in DEFAULT_DYCK_TASKS, f"Unknown task: {TASK_NAME}"

mamba_status = official_mamba_status()
mamba_status


{'installed': True,
 'package': 'mamba_ssm',
 'message': 'official mamba-ssm is available'}

In [5]:
model_specs = resolve_dyck_model_specs(
    require_official_mamba=REQUIRE_OFFICIAL_MAMBA,
    fallback_to_mamba_like=FALLBACK_TO_MAMBA_LIKE,
)
model_specs


{'rnn': {'layers': 3, 'emb_dim': 128, 'hidden_dim': 256, 'state_kind': 'h'},
 'lstm': {'layers': 3, 'emb_dim': 128, 'hidden_dim': 128, 'state_kind': 'c'},
 'transformer': {'layers': 3,
  'emb_dim': 128,
  'hidden_dim': 128,
  'n_heads': 4,
  'ffn_dim': 512,
  'state_kind': 'h'},
 'mamba': {'layers': 3,
  'emb_dim': 128,
  'hidden_dim': 128,
  'state_dim': 16,
  'expansion_factor': 2,
  'require_official_mamba': True,
  'state_kind': 'h'}}

## 4. Run Dyck

This cell trains all selected models, extracts hidden states, and runs the probes.


In [6]:
runs_df, probe_df = run_dyck_suite(
    task_name=TASK_NAME,
    seeds=SEEDS,
    training_steps=TRAINING_STEPS,
    batch_size=BATCH_SIZE,
    extract_examples=EXTRACT_EXAMPLES,
    extract_batch_size=EXTRACT_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    eval_every=EVAL_EVERY,
    device=DEVICE,
    results_root=RESULTS_ROOT,
    require_official_mamba=REQUIRE_OFFICIAL_MAMBA,
    fallback_to_mamba_like=FALLBACK_TO_MAMBA_LIKE,
)


## 5. Inspect Results

In [7]:
runs_df


,task,model,seed,run_dir,loss,accuracy,dyck_accuracy,hidden_rows,hidden_dim,label_rows
0,dyck_no_noise,rnn,0,/home/hp_twist_shan/Research/Hidden State Eval...,0.663702,0.567154,0.567154,3584,256,3584
1,dyck_no_noise,lstm,0,/home/hp_twist_shan/Research/Hidden State Eval...,0.685060,0.568484,0.568484,3584,128,3584
2,dyck_no_noise,transformer,0,/home/hp_twist_shan/Research/Hidden State Eval...,0.607694,0.605718,0.605718,3584,128,3584
3,dyck_no_noise,mamba,0,/home/hp_twist_shan/Research/Hidden State Eval...,0.730739,0.509973,0.509973,3584,128,3584


In [8]:
probe_df


,task_name,model_name,seed,left_r2,right_r2,height_r2,height_mae,cos_height_left_minus_right,cos_left_right,relevant_retention,irrelevant_decodability,irrelevant_forgetting
0,dyck_no_noise,rnn,0,0.965304,0.948449,0.990308,0.054608,1.0,0.764179,0.979446,0.0,1.0
1,dyck_no_noise,lstm,0,0.979018,0.964030,0.904184,0.299556,1.0,-0.999698,0.911337,0.0,1.0
2,dyck_no_noise,transformer,0,0.999485,0.999092,0.997620,0.035217,1.0,-0.971112,0.999366,0.0,1.0
3,dyck_no_noise,mamba,0,0.995064,0.986584,0.983799,0.098952,1.0,-0.093664,0.992328,0.0,1.0


In [9]:
sorted(str(p.relative_to(ROOT)) for p in RESULTS_ROOT.rglob("summary.json"))


['results/notebooks/dyck/dyck_no_noise/lstm_seed0/probes/summary.json',
 'results/notebooks/dyck/dyck_no_noise/mamba_seed0/probes/summary.json',
 'results/notebooks/dyck/dyck_no_noise/rnn_seed0/probes/summary.json',
 'results/notebooks/dyck/dyck_no_noise/transformer_seed0/probes/summary.json']